# Process Data
- Color Information
- Embedding Information
- Object Detection
- Captioning

In [ ]:
from Museum import Museum
from params.collections import MUSEUMS

### Color Palette

In [ ]:
for name,info in MUSEUMS.items():
  print("color:", name)
  Museum.get_colors(info)

### Embeddings (CLIP)

In [ ]:
for name,info in MUSEUMS.items():
  print("embeddings:", name)
  Museum.get_embeddings(info, "clip")

### Embeddings (SigLip2)

In [ ]:
for name,info in MUSEUMS.items():
  print("embeddings:", name)
  Museum.get_embeddings(info, "siglip2")

### Objects (Owlv2)

In [ ]:
for name,info in MUSEUMS.items():
  print("objects:", name)
  Museum.get_objects(info, "owlv2")

### Objects (Dino)

In [ ]:
for name,info in MUSEUMS.items():
  print("objects:", name)
  Museum.get_objects(info, "dino")

### Export Object Crop Images

In [ ]:
MUSEUMS = { k:v for k,v in MUSEUMS.items() if k != "file" }

for name,info in MUSEUMS.items():
  print("export crops:", name)
  Museum.export_object_crops(info, model="all")

### Captions (Llama3.2-vision)

In [ ]:
for name,info in MUSEUMS.items():
  print("caption:", name)
  Museum.get_captions(info, model="llama3.2-vision:11b")

### Captions (Gemma3)

In [ ]:
for name,info in MUSEUMS.items():
  print("caption:", name)
  Museum.get_captions(info, model="gemma3:4b")

### Combine and Export JSONs

In [ ]:
for name,info in MUSEUMS.items():
  print("combine:", name)
  Museum.combine_data(info)

## Combine Data

### Combine all museum JSONs (and add tsne embeddings)

In [ ]:
from Museum import Museum
from params.collections import MUSEUMS

MUSEUMS = { k:v for k,v in MUSEUMS.items() if k != "file" }

OUT_PREFIX = "20260801"
OUT_DIR = "./metadata/json"
DATA_TYPES = ["processed"]

Museum.combine_museums(MUSEUMS, OUT_DIR, OUT_PREFIX, DATA_TYPES, with_tsne=DATA_TYPES)

### Impute missing year

In [ ]:
import json

from Museum import Museum
from params.collections import MUSEUMS

from utils.classification_utils import impute_year

PREFIX = "20260801"
DATA_DIR = "./metadata/json"
DATA_FILE = f"{DATA_DIR}/{PREFIX}_processed.json"
DATA_FILE_OUT = f"{DATA_DIR}/{PREFIX}_processed_imputed.json"

MUSEUMS = { k:v for k,v in MUSEUMS.items() if k != "file" }
embedding_data = Museum.combine_all_data(MUSEUMS, "embeddings")

with open(DATA_FILE, "r", encoding="utf-8") as ifp:
  all_data = json.load(ifp)

imputed_data = impute_year(all_data, embedding_data)

with open(DATA_FILE_OUT, "w", encoding="utf-8") as ofp:
  json.dump(imputed_data, ofp, separators=(",",":"), sort_keys=True, ensure_ascii=False)

### Export cluster information (with cluster descriptions using Gemma3 and SigLip2)

In [ ]:
from Museum import Museum

from params.collections import MUSEUMS
from utils.data_utils import Clusterer

OUT_PREFIX = "20260801"
IMAGES_PATH = "../../imgs/arts/500"

embedding_data = Museum.combine_all_data(MUSEUMS, "embeddings")

Clusterer(embedding_data, OUT_PREFIX, IMAGES_PATH).export_clusters("clusters.json")

## Extract Activation Maps

In [ ]:
import json

from Museum import Museum

from params.collections import MUSEUMS

museum_info = { k:v for k,v in MUSEUMS.items() if k != "file" }

NCLUSTERS = 9
TCLUSTERS = "umap"

with open("./metadata/json/20260801_clusters.json", "r", encoding="utf8") as ifp:
  cdata = json.load(ifp)[f"{NCLUSTERS}"][TCLUSTERS]

for name,info in museum_info.items():
  print("activation:", name)
  Museum.get_activations(info, cdata)

### Export activation maps

In [ ]:
# RUN export_data again

## Export to HuggingFace

In [ ]:
import json
from Museum import Museum
from params.collections import MUSEUMS

from datasets import Dataset, DatasetDict
from huggingface_hub import login as hf_login

PREFIX = "20260801"
DATA_DIR = "./metadata/json"
DATA_FILE = f"{DATA_DIR}/{PREFIX}_processed.json"
CLUSTER_FILE = f"{DATA_DIR}/{PREFIX}_clusters.json"
ACTIVATION_FILE = f"{DATA_DIR}/{PREFIX}_activations.json"
NCLUSTERS = "9"
TCLUSTERS = "umap"

MUSEUMS = { k:v for k,v in MUSEUMS.items() if k != "file" }
embedding_data = Museum.combine_all_data(MUSEUMS, "embeddings")

with open(DATA_FILE, "r", encoding="utf-8") as ifp:
  all_data = json.load(ifp)

with open(ACTIVATION_FILE, "r", encoding="utf-8") as ifp:
  activation_data = json.load(ifp)

with open(CLUSTER_FILE, "r", encoding="utf-8") as ifp:
  cluster_data = json.load(ifp)
  cluster_data_images = cluster_data[NCLUSTERS][TCLUSTERS]["images"]
  cluster_data_clusters = cluster_data[NCLUSTERS][TCLUSTERS]["clusters"]
  cluster_data_clusters["n_clusters"] = NCLUSTERS
  cluster_data_clusters["dimensionality_reduction"] = TCLUSTERS

for oid in all_data.keys():
  for emb in embedding_data[oid].keys():
    all_data[oid]["embeddings"][emb] = embedding_data[oid][emb]
  all_data[oid]["clusters"] = {
    "distances": cluster_data_images[oid]["distances"],
  }
  all_data[oid]["cluster_activation"] = activation_data[oid]
  all_data[oid]["id"] = str(all_data[oid]["id"])

objects_ds = Dataset.from_list(list(all_data.values()))
clusters_ds = Dataset.from_list([cluster_data_clusters])

In [ ]:
objects_ds.push_to_hub("acervos-digitais/meta-acervos-data")
clusters_ds.push_to_hub("acervos-digitais/meta-acervos-data", "clusters")